
FAST — NUCES, Karachi Campus
<div style="border-bottom:3px solid #1F3864;padding-bottom:10px">
<h1 style="color:#1F3864;margin-bottom:2px">Lab 06 — File Handling, Exceptions and Web APIs</h1>
Department of Artificial Intelligence &amp; Data Science<br>

<i>Prepared by Khadeejah Ashraf, Lab Instructor</i>
</div>

Read the explanation, run the code cell below it with **Shift + Enter**, and compare what you
get with what you expected. Cells marked **Your turn** are for you to fill in.

**Contents**

1. Opening a file — `with` and the modes  
2. Reading text files  
3. Writing text files  
4. Exception handling  
5. Paths — where your files actually live  
6. CSV files  
7. JSON files  
8. Web APIs with `requests`  
9. A complete pipeline  
10. Class Tasks (C1 – C5)  
11. Home Tasks

> **Every file this notebook creates is written into a folder called `lab06_data`** beside the
> notebook, so nothing of yours is overwritten. Run the next cell first.
>
> **If the lab machines have no internet**, the API section falls back to a stored sample
> reply, so the whole notebook still runs from top to bottom.

In [ ]:
import os

os.makedirs('lab06_data', exist_ok=True)
os.chdir('lab06_data')       # everything below is written here

print("working folder:", os.getcwd())

working folder: /content/lab06_data


---
## 1. Opening a File — `with` and the Modes

`open()` returns a file object that must be closed. `with` closes it for you at the end of the
block, even if an error is raised inside.

| Mode | Meaning | If the file exists | If it does not |
|---|---|---|---|
| `'r'` | Read (default) | Opens it | `FileNotFoundError` |
| `'w'` | Write | **Erases it completely** | Creates it |
| `'a'` | Append | Adds at the end | Creates it |
| `'x'` | Create only | `FileExistsError` | Creates it |
| `'r+'` | Read and write | Opens it, keeps the content | `FileNotFoundError` |

In [ ]:
# create the file the rest of this notebook reads
with open('marks.txt', 'w') as f:
    f.write("Ali 78\nSara 92\nBilal 65\n")

with open('marks.txt') as f:
    data = f.read()

print(f.closed)        # True - with closed it on the way out
print(repr(data))

True
'Ali 78\nSara 92\nBilal 65\n'


In [ ]:
# 'w' erases, 'a' adds - the most expensive mistake in this lab
with open('demo.txt', 'w') as f:
    f.write("first line\n")

with open('demo.txt', 'a') as f:
    f.write("second line\n")

print(open('demo.txt').read())

with open('demo.txt', 'w') as f:     # the same file, mode 'w'
    f.write("everything above is gone\n")

print(open('demo.txt').read())

first line
second line

everything above is gone



---
## 2. Reading Text Files

Three ways to read. The `for line in f:` form holds only one line in memory at a time, so it is
the one to reach for by default.

In [ ]:
with open('marks.txt') as f:
    content = f.read()                 # the whole file as one string
print(repr(content))

with open('marks.txt') as f:
    lines = f.readlines()              # a list of lines, newlines included
print(lines)

with open('marks.txt') as f:
    for line in f:                     # one line at a time
        print(repr(line))

'Ali 78\nSara 92\nBilal 65\n'
['Ali 78\n', 'Sara 92\n', 'Bilal 65\n']
'Ali 78\n'
'Sara 92\n'
'Bilal 65\n'


### 2.1 Cleaning what you read

Every line ends with `\n`, and every value is a **string**. Stripping and converting is the
normal first step, not an optional extra.

In [ ]:
total, count = 0, 0

with open('marks.txt') as f:
    for line in f:
        line = line.strip()            # drop the newline and any spaces
        if not line:                   # skip blank lines
            continue
        name, mark = line.split()
        total += int(mark)             # convert the text into a number
        count += 1
        print(name, "scored", mark)

print("average:", round(total / count, 2))

Ali scored 78
Sara scored 92
Bilal scored 65
average: 78.33


In [ ]:
# why strip() matters
raw = "78\n"
print(raw == "78", raw.strip() == "78")
print(int(raw))          # int() happens to tolerate it...
print(len(raw), len(raw.strip()))    # ...but the string is not what you think

False True
78
3 2


**Your turn.** Write the four lines below into `cities.txt`, then read the file back and print
only the cities whose population is above 10 million.

In [ ]:
data = "Karachi 17\nLahore 13\nIslamabad 2\nFaisalabad 3\n"
# Your code here

---
## 3. Writing Text Files

`write()` adds exactly the string you give it — no newline is added for you, and a number must
be converted with `str()`.

In [ ]:
marks = {'Ali': 78, 'Sara': 92, 'Bilal': 65}

with open('report.txt', 'w') as f:
    f.write("MARKS REPORT\n")
    f.write("=" * 20 + "\n")
    for name, mark in marks.items():
        f.write(name + "," + str(mark) + "\n")     # str() is required

with open('report.txt', 'a') as f:
    f.write("end of report\n")

print(open('report.txt').read())

MARKS REPORT
Ali,78
Sara,92
Bilal,65
end of report



In [ ]:
# writelines() takes a list - the newlines must already be in the strings
rows = [f"{name},{mark}\n" for name, mark in marks.items()]

with open('report2.txt', 'w') as f:
    f.writelines(rows)

print(open('report2.txt').read())

Ali,78
Sara,92
Bilal,65



In [ ]:
# forgetting str() - run it to meet the error
try:
    with open('bad.txt', 'w') as f:
        f.write(42)
except TypeError as err:
    print("TypeError:", err)

TypeError: write() argument must be str, not int


---
## 4. Exception Handling

An exception stops the current work and looks for code prepared to deal with it. File work
raises them constantly, which is why the two topics share a lab.

`try` → `except` (only when something went wrong) → `else` (only when nothing did) →
`finally` (always).

In [ ]:
try:
    with open('missing.txt') as f:
        print(f.read())
except FileNotFoundError:
    print("that file does not exist - check the name and the folder")

print("the program carries on")

that file does not exist - check the name and the folder
the program carries on


### 4.1 Catch the exception you expect, not everything

In [ ]:
def read_number(filename):
    try:
        with open(filename) as f:
            return int(f.read().strip())
    except FileNotFoundError:
        print("no such file:", filename)
    except ValueError:
        print("the file does not contain a whole number")
    except (PermissionError, IsADirectoryError) as err:
        print("could not open it:", err)
    return None

with open('good.txt', 'w') as f:
    f.write("42")
with open('bad_number.txt', 'w') as f:
    f.write("forty two")

print(read_number('good.txt'))
print(read_number('bad_number.txt'))
print(read_number('nowhere.txt'))

42
the file does not contain a whole number
None
no such file: nowhere.txt
None


### 4.2 `as err`, `else` and `finally`

In [ ]:
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError as err:
        print("failed:", err)
        return None
    else:
        print("no error, the result is ready")
        return result
    finally:
        print("finally always runs")

print(divide(10, 4))
print(divide(10, 0))

no error, the result is ready
finally always runs
2.5
failed: division by zero
finally always runs
None


### 4.3 Raising your own

In [ ]:
class EmptyFileError(Exception):
    """Raised when a data file contains no usable rows."""

def load_marks(filename):
    with open(filename) as f:
        rows = [line.strip() for line in f if line.strip()]
    if not rows:
        raise EmptyFileError(filename + " contains no rows")
    return rows

open('empty.txt', 'w').close()        # an empty file

print(load_marks('marks.txt'))

try:
    load_marks('empty.txt')
except EmptyFileError as err:
    print("EmptyFileError:", err)

['Ali 78', 'Sara 92', 'Bilal 65']
EmptyFileError: empty.txt contains no rows


**Your turn.** Write `safe_int(text)` that returns the number, or `None` with a printed message
when the text is not a whole number. Test it on `"12"`, `"3.5"` and `"abc"`.

In [ ]:
# Your code here

---
## 5. Paths — Where Your Files Actually Live

A bare filename is relative to the folder the program runs from. Getting this wrong is the most
common cause of `FileNotFoundError`.

In [ ]:
import os

print("current folder :", os.getcwd())
print("marks.txt here?:", os.path.exists('marks.txt'))
print("full path      :", os.path.abspath('marks.txt'))
print("contents       :", sorted(os.listdir('.'))[:6])

current folder : /content/lab06_data
marks.txt here?: True
full path      : /content/lab06_data/marks.txt
contents       : ['bad.txt', 'bad_number.txt', 'demo.txt', 'empty.txt', 'good.txt', 'marks.txt']


In [ ]:
import os

# os.path.join uses the right separator for this operating system
path = os.path.join('data', 'reports', 'marks.txt')
print(path)

os.makedirs('data/reports', exist_ok=True)     # create the folders if needed

with open(path, 'w') as f:
    f.write("Ali 78\n")

print(os.path.exists(path), os.path.getsize(path), "bytes")
print(os.path.splitext(path))

# On Windows, write paths as raw strings: r'C:\Users\you\data.txt'

data/reports/marks.txt
True 7 bytes
('data/reports/marks', '.txt')


---
## 6. CSV Files

A table stored as text. Use the `csv` module rather than `split(',')` — a value that *contains*
a comma breaks hand-rolled splitting immediately.

In [ ]:
import csv

rows = [['name', 'subject', 'mark'],
        ['Ali', 'Physics', 78],
        ['Sara', 'Maths', 92],
        ['Khan, Ali', 'Chemistry', 65]]      # note the comma inside a value

with open('marks.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print(open('marks.csv').read())

name,subject,mark
Ali,Physics,78
Sara,Maths,92
"Khan, Ali",Chemistry,65



In [ ]:
# what hand-splitting does to that third student
with open('marks.csv') as f:
    next(f)
    for line in f:
        print(line.strip().split(','))       # 'Khan, Ali' has been torn in two

['Ali', 'Physics', '78']
['Sara', 'Maths', '92']
['"Khan', ' Ali"', 'Chemistry', '65']


In [ ]:
import csv

with open('marks.csv', newline='') as f:
    reader = csv.reader(f)
    header = next(reader)                    # pull the header off first
    print("columns:", header)
    for row in reader:
        print(row, "-> mark as a number:", int(row[2]))

columns: ['name', 'subject', 'mark']
['Ali', 'Physics', '78'] -> mark as a number: 78
['Sara', 'Maths', '92'] -> mark as a number: 92
['Khan, Ali', 'Chemistry', '65'] -> mark as a number: 65


### 6.1 `DictReader` and `DictWriter` — rows as dictionaries

In [ ]:
import csv

with open('marks.csv', newline='') as f:
    reader = csv.DictReader(f)               # the header becomes the keys
    total = 0
    for row in reader:
        print(row['name'], "->", row['mark'])
        total += int(row['mark'])
print("total:", total)

Ali -> 78
Sara -> 92
Khan, Ali -> 65
total: 235


In [ ]:
import csv

records = [{'name': 'Hina', 'subject': 'Maths', 'mark': 88},
           {'name': 'Usman', 'subject': 'Physics', 'mark': 54}]

with open('marks2.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'subject', 'mark'])
    writer.writeheader()
    writer.writerows(records)

print(open('marks2.csv').read())

name,subject,mark
Hina,Maths,88
Usman,Physics,54



**Your turn.** Read `marks.csv` with `DictReader` and write a new file `passed.csv` containing
only the rows whose mark is 70 or more, header included.

In [ ]:
# Your code here

---
## 7. JSON Files

JSON is text that represents nested data — and it is what almost every web API speaks.

| Function | Direction | Works on |
|---|---|---|
| `json.dump(obj, f)` | Python → JSON | an open file |
| `json.load(f)` | JSON → Python | an open file |
| `json.dumps(obj)` | Python → JSON | returns a string |
| `json.loads(text)` | JSON → Python | reads a string |

The pair with the **s** works on **s**trings. That is the whole distinction.

In [ ]:
import json

students = [
    {'name': 'Ali',  'marks': [78, 45, 90], 'active': True},
    {'name': 'Sara', 'marks': [88, 92, 95], 'active': False},
]

with open('students.json', 'w') as f:
    json.dump(students, f, indent=2)

print(open('students.json').read())

[
  {
    "name": "Ali",
    "marks": [
      78,
      45,
      90
    ],
    "active": true
  },
  {
    "name": "Sara",
    "marks": [
      88,
      92,
      95
    ],
    "active": false
  }
]


In [ ]:
import json

with open('students.json') as f:
    loaded = json.load(f)

print(type(loaded), len(loaded))
print(loaded[0]['name'], loaded[0]['marks'])
print(loaded[1]['active'], type(loaded[1]['active']))

<class 'list'> 2
Ali [78, 45, 90]
False <class 'bool'>


In [ ]:
import json

text = json.dumps({'name': 'Ali', 'cgpa': 3.4})     # to a string
print(text, type(text))

back = json.loads(text)                             # and back again
print(back['cgpa'] + 0.1)

{"name": "Ali", "cgpa": 3.4} <class 'str'>
3.5


In [ ]:
import json

# what survives a round trip, and what does not
original = {'tuple': (1, 2), 'int_key': {1: 'a'}, 'none': None}
restored = json.loads(json.dumps(original))
print(restored)
print(type(restored['tuple']), list(restored['int_key'].keys()))

{'tuple': [1, 2], 'int_key': {'1': 'a'}, 'none': None}
<class 'list'> ['1']


### 7.1 When JSON goes wrong

In [ ]:
import json

with open('broken.json', 'w') as f:
    f.write('{"name": "Ali", }')          # a trailing comma is not valid JSON

try:
    with open('broken.json') as f:
        json.load(f)
except json.JSONDecodeError as err:
    print("JSONDecodeError:", err)

try:
    json.dumps({'subjects': {'Physics', 'Maths'}})    # a set cannot be stored
except TypeError as err:
    print("TypeError:", err)

JSONDecodeError: Illegal trailing comma before end of object: line 1 column 15 (char 14)
TypeError: Object of type set is not JSON serializable


---
## 8. Web APIs with `requests`

Your program sends a request to a URL; the server sends back a status code and, usually, a JSON
body. `requests` is not in the standard library — install it once with `pip install requests`.

**Status codes:** `2xx` success · `3xx` redirect · `4xx` your mistake (`404` not found, `401`
no key, `429` too many requests) · `5xx` the server's problem.

The cell below decides once whether this machine can reach the internet. Everything after it
works either way.

In [ ]:
import json

SAMPLE = {
    "login": "python", "name": "Python", "public_repos": 24,
    "created_at": "2008-07-23T14:17:49Z", "location": "Netherlands",
    "blog": "https://www.python.org/", "followers": 20000,
}

try:
    import requests
    probe = requests.get("https://api.github.com/users/python", timeout=8)
    ONLINE = probe.status_code == 200
except Exception as err:
    ONLINE = False
    print("offline mode:", type(err).__name__)

print("ONLINE =", ONLINE)

ONLINE = True


In [ ]:
# a first request - or the stored sample if this machine is offline
if ONLINE:
    r = requests.get("https://api.github.com/users/python", timeout=10)
    print("status code :", r.status_code)
    print("content type:", r.headers['Content-Type'])
    data = r.json()               # JSON text -> a Python dictionary
else:
    data = SAMPLE
    print("using the stored sample reply")

print(type(data))
print(data['login'], "-", data['public_repos'], "public repos")

status code : 200
content type: application/json; charset=utf-8
<class 'dict'>
python - 95 public repos


### 8.1 Check the status code *before* parsing the body

In [ ]:
if ONLINE:
    r = requests.get("https://api.github.com/users/this-user-does-not-exist-xyz",
                     timeout=10)
    print(r.status_code, r.ok)
    if r.status_code == 200:
        print(r.json()['login'])
    else:
        print("the request failed, so the body was not parsed")
else:
    print("skipped - no network")

404 False
the request failed, so the body was not parsed


### 8.2 A function that handles every way a request can fail

In [ ]:
def fetch_user(username, timeout=10):
    """Return the user dictionary, or None if anything goes wrong."""
    if not ONLINE:
        return SAMPLE if username == 'python' else None

    url = "https://api.github.com/users/" + username
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()          # turns 4xx and 5xx into an exception
        return r.json()
    except requests.exceptions.Timeout:
        print("the server took too long to answer")
    except requests.exceptions.ConnectionError:
        print("no network connection")
    except requests.exceptions.HTTPError as err:
        print("refused:", err.response.status_code, "for", username)
    except json.JSONDecodeError:
        print("the reply was not JSON")
    return None


for name in ['python', 'this-user-does-not-exist-xyz']:
    user = fetch_user(name)
    print(name, "->", (user['public_repos'] if user else "nothing to show"))

python -> 95
refused: 404 for this-user-does-not-exist-xyz
this-user-does-not-exist-xyz -> nothing to show


**Be a good citizen.** A public API is someone else's machine: set a `timeout`, never put a
bare request inside a fast loop, add a short `time.sleep()` between calls, and cache the reply
to a file while you are developing so a hundred test runs cost one request.

---
## 9. A Complete Pipeline — Fetch, Parse, Store

Fetch the data, survive the failures, keep the raw reply as JSON, write the tidy summary as
CSV. Only `main()` prints.

In [ ]:
import csv, json


def collect(names):
    """Return a list of user dictionaries, skipping any that fail."""
    found = []
    for name in names:
        user = fetch_user(name)
        if user:
            found.append(user)
    return found


def save_raw(users, filename='raw_users.json'):
    """Keep the untouched reply so the API need not be called again."""
    with open(filename, 'w') as f:
        json.dump(users, f, indent=2)


def summarise(users):
    """Pull out only the fields we care about."""
    return [{'login': u['login'],
             'repos': u['public_repos'],
             'created': u['created_at'][:10]} for u in users]


def save_csv(rows, filename='users.csv'):
    with open(filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['login', 'repos', 'created'])
        writer.writeheader()
        writer.writerows(rows)


def main():
    users = collect(['python', 'no-such-user-xyz'])
    if not users:
        print("nothing was fetched")
        return
    save_raw(users)
    rows = summarise(users)
    save_csv(rows)
    for row in sorted(rows, key=lambda r: r['repos'], reverse=True):
        print(row['login'], row['repos'], row['created'])


main()
print()
print(open('users.csv').read())

refused: 404 for no-such-user-xyz
python 95 2012-03-11

login,repos,created
python,95,2012-03-11



---
# 10. Class Tasks

Complete all five during the lab session and demonstrate them to the instructor before you
leave. Every file operation must use `with`, every `except` must name the exception it guards
against, and every task must survive being run twice in a row. Run
**Kernel ▸ Restart & Run All** before you call the instructor.

### C1 — A Text-File Report
*reading · writing · cleaning*

Create `students.txt` from your own code with at least six lines of the form
`name,subject,mark` — and deliberately include a blank line, a line with a missing field, and a
line whose mark is not a number.

Read it back line by line and print: the number of lines, the number of **valid** records, the
overall average mark, the highest and lowest scorer, and every line that had to be rejected.
Catch each failure with the right exception, not with a bare `except`.

Finish by writing the summary to `report_c1.txt` in **append** mode, so that running the
program again adds a second report instead of erasing the first.

In [ ]:
# C1 - A Text-File Report
# Name:
# Roll number:

# Your code here

### C2 — A Robust Reader
*exception handling*

Write `load_marks(filename)` returning a list of `(name, mark)` tuples, handling separately:

* `FileNotFoundError` — report it and return an empty list
* `ValueError` — skip that one line, keep going, and count how many were skipped
* `PermissionError`
* your own `EmptyFileError(Exception)`, raised when the file parses but holds no usable rows

Use `finally` to print how many lines were examined, whatever happened. Call the function four
times — a good file, a file with bad lines, an empty file, and a name that does not exist — and
make sure all four run one after another without the notebook stopping.

In [ ]:
# C2 - A Robust Reader
# Name:
# Roll number:

# Your code here

### C3 — Marks to CSV and Back
*the csv module*

Build at least six student dictionaries with the keys `roll`, `name`, `subject`, `mark` — one
name must contain a comma, such as `"Khan, Ali"` — and write them to `marks_c3.csv` with
`DictWriter`, header included.

Reading the file back with `DictReader` (never with `split(',')`), print the class average, the
number of students per subject using a dictionary, and the top scorer in each subject. Then
write `toppers.csv` containing only those top scorers.

Show that the comma-containing name survived the round trip, and explain in a comment why
splitting on commas by hand would have broken.

In [ ]:
# C3 - Marks to CSV and Back
# Name:
# Roll number:

# Your code here

### C4 — A JSON Address Book
*nested data*

Build a dictionary of at least four people, each name mapping to a dictionary with `age`, a
list of `subjects`, and a nested `contact` dictionary holding `email` and `city`. Save it to
`people.json` with `indent=2`, load it back, and print everyone's name and city, the oldest
person, and anyone who shares that age. Then add a new person, save again, and reload to prove
the addition persisted.

Also demonstrate two failures inside `try` / `except`: loading a deliberately corrupted JSON
file, and calling `json.dumps` on a dictionary containing a set.

In [ ]:
# C4 - A JSON Address Book
# Name:
# Roll number:

# Your code here

### C5 — A Live API Client
*requests end to end*

Write `fetch_user(username)` against `https://api.github.com/users/<username>` that returns the
user dictionary or `None`, checking the status code and catching `Timeout`, `ConnectionError`
and `HTTPError` **separately**.

Call it for three usernames, one of which must not exist. For every user found, print the login
name, the number of public repositories and the creation date; save the full replies to
`users_c5.json`; and write a summary table of the found users to `users_c5.csv`.

If this machine has no internet, use the `SAMPLE` dictionary defined earlier so that your
parsing, saving and error handling can still be demonstrated in full — your code must not
change between the two cases apart from where the data comes from.

In [ ]:
# C5 - A Live API Client
# Name:
# Roll number:

# Your code here

---
# 11. Home Tasks

These are **not** done in this notebook. Write each one in a separate script named
`RollNo_Lab06_Qn.py` (for example `22K-1234_Lab06_Q7.py`), handle every exception that can
reasonably occur, submit the data files alongside the code, and begin every file with a comment
giving your name, roll number and question number.

1. Read a text file and print the number of characters, words and lines in it. Handle every possible exception.
2. Read a text file, replace every occurrence of a word or phrase given by the user with another, write the result back to the same file, and report how many replacements were made.
3. Take two lists of equal length from the user, build a dictionary pairing them element by element — **your own logic, no `zip()` and no library** — and store that dictionary in a text file.
4. Take an employee's biodata (name, CNIC number, age, salary) and save it to a text file; then **append** a contact number; then read the whole file back and print it. Explain in a comment what mode `'w'` would have done at the second step.
5. Take name-and-age pairs such as `{'Ali': 23, 'Saad': 24, 'Salman': 15, 'Shams': 25, 'Sadiq': 46, 'Hammad': 23}`, save them as JSON, load them back, and print the oldest person and anyone who shares an age — working out the maximum and the duplicates with your own loops, not with `max()` or `Counter`.
6. Write a function that asks for a sentence and appends it to `questions.txt` only if it is a question. Call it in a loop until the user types `quit`, then print the whole file.
7. `replacement_needed.txt` contains one sentence with a single mistake: one letter must be replaced for it to make sense. Read it, work out and perform the replacement with your own logic, and write the corrected sentence to a new file. Put your code in a function and call it.
8. Take two numbers from the user and divide them, handling `ZeroDivisionError` and `ValueError`. Keep asking until both inputs are valid, then report the result.
9. Read a CSV of students (`roll,name,subject,mark`) with `DictReader` and write two new CSV files — one of those who passed (50 or more) and one of those who failed. Print how many rows went into each, and handle a missing file and a malformed row.
10. Convert between formats: read `students.csv`, write the same data as `students.json`, read that JSON back, and write it out as a plain text report with one line per student. Verify the record count is identical at every stage.
11. Write a log analyser. Create a text file whose lines are `date time level message` with levels `INFO`, `WARNING`, `ERROR`. Report how many lines carried each level, the first and last `ERROR` message, and every line for a date entered by the user. Handle lines that do not match the expected shape.
12. Using `requests`, fetch a public API that needs no key, check the status code, parse the JSON, print three interesting fields, and save the full reply to a file. Catch `Timeout`, `ConnectionError` and `HTTPError` separately, and print a clear message rather than crashing when the machine is offline.

---
<div style="text-align:center;color:#666"><i>End of Lab 06 — FAST NUCES, Karachi Campus</i></div>